# LAB 2 — Step-Back Prompting
## Aula 7 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Implementar o Step-Back Prompting para transformar queries jurídicas hiper-específicas em abstrações principiológicas, comparando a qualidade do retrieval antes e depois da transformação em 5 perguntas conceituais.

**Tempo estimado:** 35 minutos

---
**Checklist de entrega:**
- [ ] Prompt de abstração implementado com vLLM
- [ ] Tabela com 5 pares: query original → query abstraída
- [ ] Comparação de retrieval: antes vs depois do Step-Back
- [ ] Identificação de pelo menos 1 caso onde Step-Back não ajuda

In [ ]:
!pip install -q langchain langchain-openai sentence-transformers opensearch-py pandas matplotlib
print('✅ Dependências instaladas!')

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer
from opensearchpy import OpenSearch

# Configurações
VLLM_BASE_URL = os.getenv('VLLM_BASE_URL', 'http://localhost:8000/v1')
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
INDEX_NAME = 'corpus_juridico_aula7'

# Clientes
llm = ChatOpenAI(
    base_url=VLLM_BASE_URL, api_key='dummy',
    model='meta-llama/Llama-3.1-8B-Instruct',
    temperature=0.3  # temperatura baixa para abstrações mais consistentes
)

embeddings = SentenceTransformer('BAAI/bge-m3')

os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': 9200}],
    http_auth=('admin', os.getenv('OPENSEARCH_PASS', 'Admin123!')),
    use_ssl=False, verify_certs=False
)

print('✅ Ambiente configurado!')

## 2. Prompt de Abstração Step-Back

In [ ]:
# ── Prompt Step-Back especializado para domínio jurídico ──────────────
STEP_BACK_PROMPT = PromptTemplate(
    input_variables=['query'],
    template="""Você é um jurista especializado em recuperação de informações jurídicas.

Dada a pergunta abaixo (que pode ser muito específica, com nomes, datas, artigos de lei
ou números de processo), formule UMA pergunta mais geral que capture os princípios jurídicos
subjacentes sem depender dos detalhes específicos do caso.

A pergunta geral deve:
- Ser respondível por documentos doutrinários, normativos ou jurisprudenciais gerais
- Remover referências a pessoas, datas, locais e processos específicos
- Preservar o tema jurídico central (ex: prisão, contrato, prova, etc.)
- Ser formulada de forma que um livro de direito processual penal ou civil pudesse respondê-la

Retorne APENAS a pergunta abstraída, sem prefixos ou explicações.

Pergunta original: {query}

Pergunta abstraída (Step-Back):"""
)

def step_back(query: str) -> str:
    """Transforma uma query específica em uma abstração principiológica."""
    response = llm.invoke(STEP_BACK_PROMPT.format(query=query))
    return response.content.strip()

# Teste rápido
query_exemplo = 'O réu Marcos Souza, preso em flagrante em São Paulo em 15/01/2024 por tráfico de drogas segundo o art. 33 da Lei 11.343/06, pode ter prisão convertida em medida cautelar alternativa?'
abstract = step_back(query_exemplo)
print(f'Original: {query_exemplo}')
print(f'\nStep-Back: {abstract}')

## 3. Função de Retrieval

In [ ]:
def retrieve(query: str, k: int = 5) -> list:
    """Executa retrieval kNN no OpenSearch para uma query."""
    q_vec = embeddings.encode(query, normalize_embeddings=True).tolist()
    
    resp = os_client.search(
        index=INDEX_NAME,
        body={
            'size': k,
            'query': {'knn': {'embedding': {'vector': q_vec, 'k': k}}},
            '_source': ['content', 'source', 'tipo_documento']
        }
    )
    
    return [
        {
            'id': h['_id'],
            'score': h['_score'],
            'content': h['_source'].get('content', '')[:150],
            'source': h['_source'].get('source', 'N/A'),
            'tipo': h['_source'].get('tipo_documento', 'N/A')
        }
        for h in resp['hits']['hits']
    ]

def relevance_score_heuristic(docs: list, keywords: list) -> float:
    """
    Heurística simples de relevância: % dos documentos que contêm
    ao menos uma das palavras-chave esperadas.
    """
    if not docs:
        return 0.0
    relevantes = sum(
        1 for doc in docs
        if any(kw.lower() in doc['content'].lower() for kw in keywords)
    )
    return relevantes / len(docs)

print('✅ Funções de retrieval e avaliação definidas!')

## 4. Comparação em 5 Queries Jurídicas Conceituais

In [ ]:
# Queries: (query_original, keywords_esperadas)
casos = [
    (
        'O réu Marcos Souza, preso em 15/01/2024 por art. 33 Lei 11.343/06, pode ter prisão convertida em cautelar alternativa?',
        ['medida cautelar', 'prisão preventiva', 'tráfico', 'alternativa à prisão']
    ),
    (
        'A vítima Maria, agredida por seu marido João em 20/03/2023, pode requerer medida protetiva sem advogado?',
        ['medida protetiva', 'Lei Maria da Penha', 'violência doméstica', 'tutela']
    ),
    (
        'O processo 1234567-89.2023.8.26.0100 da 5ª Vara Criminal de São Paulo está prescrito após 4 anos?',
        ['prescrição', 'prazo prescricional', 'extinção da punibilidade']
    ),
    (
        'Delegado da 3ª DP de Campinas pode realizar busca e apreensão no celular do suspeito preso em flagrante sem mandado?',
        ['busca e apreensão', 'inviolabilidade', 'sigilo', 'mandado', 'flagrante']
    ),
    (
        'A Súmula Vinculante 11 do STF se aplica ao caso do policial militar que algemou manifestante em protesto em Brasília em 2023?',
        ['Súmula Vinculante 11', 'algemas', 'uso proporcional', 'força policial']
    )
]

resultados = []

for i, (query_orig, keywords) in enumerate(casos, 1):
    print(f'\n{'='*60}')
    print(f'CASO {i}: {query_orig[:80]}...')
    print('='*60)
    
    # Step-Back
    query_abstrata = step_back(query_orig)
    print(f'Step-Back: {query_abstrata}')
    
    # Retrieval original
    docs_orig = retrieve(query_orig, k=5)
    relevancia_orig = relevance_score_heuristic(docs_orig, keywords)
    
    # Retrieval com Step-Back
    docs_sb = retrieve(query_abstrata, k=5)
    relevancia_sb = relevance_score_heuristic(docs_sb, keywords)
    
    print(f'\n  Relevância Original:   {relevancia_orig:.0%} ({relevancia_orig*5:.0f}/5 docs relevantes)')
    print(f'  Relevância Step-Back:  {relevancia_sb:.0%} ({relevancia_sb*5:.0f}/5 docs relevantes)')
    
    delta = relevancia_sb - relevancia_orig
    melhora = '📈 Melhora' if delta > 0 else ('🔄 Igual' if delta == 0 else '📉 Piora')
    print(f'  Delta: {delta:+.0%} → {melhora}')
    
    resultados.append({
        'caso': i,
        'query_original': query_orig[:60] + '...',
        'query_abstrata': query_abstrata[:60] + '...',
        'relevancia_original': relevancia_orig,
        'relevancia_step_back': relevancia_sb,
        'delta': delta
    })

## 5. Visualização dos Resultados

In [ ]:
df = pd.DataFrame(resultados)

print('\n=== TABELA COMPARATIVA: Step-Back Prompting ===')
print(df[['caso', 'relevancia_original', 'relevancia_step_back', 'delta']].to_string(index=False))

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Relevância antes vs depois
x = range(len(resultados))
width = 0.35
axes[0].bar([i - width/2 for i in x], df['relevancia_original'], width,
            label='Sem Step-Back', color='#e74c3c', alpha=0.8)
axes[0].bar([i + width/2 for i in x], df['relevancia_step_back'], width,
            label='Com Step-Back', color='#2ecc71', alpha=0.8)
axes[0].set_title('Relevância dos Documentos Recuperados\n(Heurística: % com keywords)', fontsize=11)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Caso {i+1}' for i in x], rotation=15)
axes[0].set_ylabel('Score de Relevância')
axes[0].set_ylim(0, 1.1)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Gráfico 2: Delta
cores = ['#2ecc71' if d > 0 else ('#e74c3c' if d < 0 else '#95a5a6') for d in df['delta']]
axes[1].bar(x, df['delta'], color=cores, alpha=0.8)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_title('Ganho/Perda de Relevância\ncom Step-Back', fontsize=11)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Caso {i+1}' for i in x], rotation=15)
axes[1].set_ylabel('Delta de Relevância')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df['delta']):
    axes[1].text(i, v + 0.005, f'{v:+.0%}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('step_back_comparacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Quando Step-Back NÃO Ajuda

In [ ]:
# Casos onde Step-Back pode prejudicar o retrieval
casos_negativos = [
    (
        'Qual é o teor exato da Súmula Vinculante nº 11 do STF sobre uso de algemas?',
        'Busca por texto específico de súmula — abstração perde o alvo'
    ),
    (
        'Quais são os princípios gerais do processo penal?',
        'Query já é abstrata — Step-Back sobre-generaliza'
    ),
    (
        'O que diz o artigo 5º, inciso LVIII, da Constituição Federal?',
        'Busca de referência normativa específica — abstração não faz sentido'
    )
]

print('=== CASOS ONDE STEP-BACK NÃO AJUDA ===')
for query, motivo in casos_negativos:
    abstrata = step_back(query)
    print(f'\n  Original: {query}')
    print(f'  Abstrata: {abstrata}')
    print(f'  ⚠️  Motivo: {motivo}')

print('\n💡 CONCLUSÃO: Step-Back é mais eficaz quando a query contém:')
print('   ✅ Nomes próprios de pessoas ou lugares')
print('   ✅ Datas e números de processo específicos')
print('   ✅ Artigos de lei com inciso/alínea específicos de um caso concreto')
print('   ❌ NÃO usar quando a query já é abstrata ou busca texto normativo exato')

In [ ]:
print('=== CHECKLIST DE ENTREGA — LAB 2 ===')
checklist = [
    'Prompt Step-Back implementado com vLLM',
    'Tabela com 5 pares: query original → abstrata',
    'Comparação de relevância documentada (antes vs depois)',
    'Visualização gerada e salva',
    'Análise de casos onde Step-Back não ajuda (mínimo 1 caso)'
]
for item in checklist:
    print(f'  [ ] {item}')
print('\n✅ Lab 2 concluído! Prossiga para o LAB 3 — RAG-Fusion.')